# Model Context Protocol (MCP)

**Module 11: AI Agents and MCP**  
**Objective**: Standardized protocol for AI-tool integration

Model Context Protocol (MCP):
- Universal standard for connecting AI systems to data/tools
- Developed by Anthropic
- Server-client architecture
- Tool registration and discovery
- Structured data exchange

## What You'll Learn
1. MCP architecture and design
2. Building MCP servers
3. MCP client implementation
4. Tool/resource registration
5. Real-world integrations
6. Security and best practices

In [ ]:
import json
import asyncio
from typing import Dict, List, Any, Callable, Optional
from dataclasses import dataclass, asdict
from enum import Enum
import hashlib
from datetime import datetime

# Simulated imports (in production, use official MCP SDK)
# from mcp import Server, Client, Tool, Resource

print("Model Context Protocol (MCP) Implementation\\n")

## 1. What is MCP?

**Model Context Protocol** = Universal standard for AI-tool integration

**Problem it solves**:
- Every app has custom AI integration
- LLMs can't easily access external data
- No standard for tool calling

**MCP Solution**:
- **Servers** expose tools/resources
- **Clients** (AI apps) discover and use them
- **Protocol** defines communication standard

**Architecture**:
```
AI Application (Claude, ChatGPT, etc.)
         │
         │ MCP Client
         │
         ├─────┬─────┬─────┐
         │     │     │     │
      Server Server Server Server
         │     │     │     │
    Database Files API  CRM
```

In [ ]:
class MCPArchitecture:
    """Visualize MCP architecture."""
    
    @staticmethod
    def print_architecture():
        print("="*70)
        print("MODEL CONTEXT PROTOCOL ARCHITECTURE")
        print("="*70)
        
        print("""
┌──────────────────────────────────────────────────────┐
│           AI Application (Claude, ChatGPT)           │
│                                                      │
│  ┌────────────────────────────────────────────────┐ │
│  │           MCP Client (Built-in)                │ │
│  │                                                │ │
│  │  • Tool Discovery                              │ │
│  │  • Resource Access                             │ │
│  │  • Execution Management                        │ │
│  └────────────────────────────────────────────────┘ │
└───────────────────┬──────────────────────────────────┘
                    │
        ┌───────────┼───────────┬───────────┐
        │           │           │           │
    ┌───▼────┐  ┌───▼────┐  ┌───▼────┐  ┌───▼────┐
    │ Server │  │ Server │  │ Server │  │ Server │
    │        │  │        │  │        │  │        │
    │ Database│  │ Files  │  │  API   │  │  CRM   │
    └────────┘  └────────┘  └────────┘  └────────┘

**Key Benefits**:
1. Standardization: One protocol for all tools
2. Discoverability: Automatic tool detection
3. Security: Controlled access patterns
4. Composability: Mix-and-match servers
        """)
        
        print("\\nExamples:")
        print("  • Database Server: Query PostgreSQL, read tables")
        print("  • File Server: Read/write local files")
        print("  • API Server: Call REST APIs (GitHub, Slack, etc.)")
        print("  • Custom Server: Your app-specific tools")

MCPArchitecture.print_architecture()

## 2. MCP Protocol Spec

**Core Concepts**:

1. **Tools**: Functions AI can call
2. **Resources**: Data AI can read (files, DB records)
3. **Prompts**: Templates for common workflows
4. **Sampling**: LLM completions

**Message Types**:
- `initialize`: Handshake
- `tools/list`: Get available tools
- `tools/call`: Execute tool
- `resources/list`: Get resources
- `resources/read`: Read resource content

In [ ]:
# MCP Protocol Message Structures

class MessageType(Enum):
    """MCP message types."""
    INITIALIZE = "initialize"
    TOOLS_LIST = "tools/list"
    TOOLS_CALL = "tools/call"
    RESOURCES_LIST = "resources/list"
    RESOURCES_READ = "resources/read"
    PROMPTS_LIST = "prompts/list"
    PROMPTS_GET = "prompts/get"

@dataclass
class MCPMessage:
    """Base MCP message."""
    jsonrpc: str = "2.0"
    id: Optional[int] = None
    method: Optional[str] = None
    params: Optional[Dict] = None
    result: Optional[Any] = None
    error: Optional[Dict] = None
    
    def to_json(self) -> str:
        return json.dumps(asdict(self), indent=2)

@dataclass
class ToolDefinition:
    """MCP tool definition."""
    name: str
    description: str
    inputSchema: Dict  # JSON Schema
    
    def to_dict(self) -> Dict:
        return asdict(self)

@dataclass
class ResourceDefinition:
    """MCP resource definition."""
    uri: str
    name: str
    description: str
    mimeType: str
    
    def to_dict(self) -> Dict:
        return asdict(self)

# Example tool definition
calculator_tool = ToolDefinition(
    name="calculate",
    description="Perform mathematical calculations",
    inputSchema={
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "Math expression to evaluate"
            }
        },
        "required": ["expression"]
    }
)

print("Example Tool Definition:")
print(json.dumps(calculator_tool.to_dict(), indent=2))

# Example resource definition
database_resource = ResourceDefinition(
    uri="db://localhost/users",
    name="Users Database",
    description="Customer user records",
    mimeType="application/json"
)

print("\\nExample Resource Definition:")
print(json.dumps(database_resource.to_dict(), indent=2))

# Example messages
init_message = MCPMessage(
    id=1,
    method="initialize",
    params={
        "protocolVersion": "2024-11-05",
        "capabilities": {
            "tools": True,
            "resources": True,
            "prompts": False
        },
        "clientInfo": {
            "name": "my-mcp-client",
            "version": "1.0.0"
        }
    }
)

print("\\nExample Initialize Message:")
print(init_message.to_json())

list_tools_message = MCPMessage(
    id=2,
    method="tools/list",
    params={}
)

print("\\nExample Tools List Request:")
print(list_tools_message.to_json())

## 3. Building an MCP Server

MCP servers expose tools/resources to AI apps.

**Server responsibilities**:
- Register tools/resources
- Handle tool execution
- Manage authentication
- Return structured responses

In [ ]:
class MCPServer:
    """
    Simple MCP server implementation.
    
    Exposes tools and resources following MCP protocol.
    """
    
    def __init__(self, name: str, version: str = "1.0.0"):
        self.name = name
        self.version = version
        self.tools: Dict[str, Dict] = {}
        self.tool_handlers: Dict[str, Callable] = {}
        self.resources: Dict[str, Dict] = {}
        self.resource_handlers: Dict[str, Callable] = {}
        self.initialized = False
        
    def register_tool(self, tool_def: ToolDefinition, handler: Callable):
        """Register a tool with its handler function."""
        self.tools[tool_def.name] = tool_def.to_dict()
        self.tool_handlers[tool_def.name] = handler
        print(f"✓ Registered tool: {tool_def.name}")
    
    def register_resource(self, resource_def: ResourceDefinition, handler: Callable):
        """Register a resource with its handler function."""
        self.resources[resource_def.uri] = resource_def.to_dict()
        self.resource_handlers[resource_def.uri] = handler
        print(f"✓ Registered resource: {resource_def.name} ({resource_def.uri})")
    
    def handle_message(self, message: Dict) -> Dict:
        """Handle incoming MCP message."""
        method = message.get("method")
        msg_id = message.get("id")
        params = message.get("params", {})
        
        if method == "initialize":
            return self._handle_initialize(msg_id, params)
        elif method == "tools/list":
            return self._handle_tools_list(msg_id)
        elif method == "tools/call":
            return self._handle_tools_call(msg_id, params)
        elif method == "resources/list":
            return self._handle_resources_list(msg_id)
        elif method == "resources/read":
            return self._handle_resources_read(msg_id, params)
        else:
            return {
                "jsonrpc": "2.0",
                "id": msg_id,
                "error": {
                    "code": -32601,
                    "message": f"Method not found: {method}"
                }
            }
    
    def _handle_initialize(self, msg_id: int, params: Dict) -> Dict:
        """Handle initialize request."""
        self.initialized = True
        return {
            "jsonrpc": "2.0",
            "id": msg_id,
            "result": {
                "protocolVersion": "2024-11-05",
                "capabilities": {
                    "tools": len(self.tools) > 0,
                    "resources": len(self.resources) > 0,
                    "prompts": False
                },
                "serverInfo": {
                    "name": self.name,
                    "version": self.version
                }
            }
        }
    
    def _handle_tools_list(self, msg_id: int) -> Dict:
        """List available tools."""
        return {
            "jsonrpc": "2.0",
            "id": msg_id,
            "result": {
                "tools": list(self.tools.values())
            }
        }
    
    def _handle_tools_call(self, msg_id: int, params: Dict) -> Dict:
        """Execute a tool."""
        tool_name = params.get("name")
        arguments = params.get("arguments", {})
        
        if tool_name not in self.tool_handlers:
            return {
                "jsonrpc": "2.0",
                "id": msg_id,
                "error": {
                    "code": -32602,
                    "message": f"Tool not found: {tool_name}"
                }
            }
        
        try:
            handler = self.tool_handlers[tool_name]
            result = handler(**arguments)
            
            return {
                "jsonrpc": "2.0",
                "id": msg_id,
                "result": {
                    "content": [
                        {
                            "type": "text",
                            "text": str(result)
                        }
                    ]
                }
            }
        except Exception as e:
            return {
                "jsonrpc": "2.0",
                "id": msg_id,
                "error": {
                    "code": -32603,
                    "message": f"Tool execution error: {str(e)}"
                }
            }
    
    def _handle_resources_list(self, msg_id: int) -> Dict:
        """List available resources."""
        return {
            "jsonrpc": "2.0",
            "id": msg_id,
            "result": {
                "resources": list(self.resources.values())
            }
        }
    
    def _handle_resources_read(self, msg_id: int, params: Dict) -> Dict:
        """Read resource content."""
        uri = params.get("uri")
        
        if uri not in self.resource_handlers:
            return {
                "jsonrpc": "2.0",
                "id": msg_id,
                "error": {
                    "code": -32602,
                    "message": f"Resource not found: {uri}"
                }
            }
        
        try:
            handler = self.resource_handlers[uri]
            content = handler()
            
            return {
                "jsonrpc": "2.0",
                "id": msg_id,
                "result": {
                    "contents": [
                        {
                            "uri": uri,
                            "mimeType": self.resources[uri]["mimeType"],
                            "text": content
                        }
                    ]
                }
            }
        except Exception as e:
            return {
                "jsonrpc": "2.0",
                "id": msg_id,
                "error": {
                    "code": -32603,
                    "message": f"Resource read error: {str(e)}"
                }
            }

# Create server
server = MCPServer(name="demo-server", version="1.0.0")

# Define and register tools
def calculator_handler(expression: str) -> str:
    """Calculate mathematical expression."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {str(e)}"

def weather_handler(location: str) -> str:
    """Get weather (simulated)."""
    weather_db = {
        "San Francisco": "Sunny, 72°F",
        "New York": "Cloudy, 65°F",
        "London": "Rainy, 55°F"
    }
    return weather_db.get(location, "Weather data not available")

calculator_tool_def = ToolDefinition(
    name="calculate",
    description="Perform mathematical calculations",
    inputSchema={
        "type": "object",
        "properties": {
            "expression": {"type": "string", "description": "Expression to evaluate"}
        },
        "required": ["expression"]
    }
)

weather_tool_def = ToolDefinition(
    name="get_weather",
    description="Get current weather for a location",
    inputSchema={
        "type": "object",
        "properties": {
            "location": {"type": "string", "description": "City name"}
        },
        "required": ["location"]
    }
)

server.register_tool(calculator_tool_def, calculator_handler)
server.register_tool(weather_tool_def, weather_handler)

# Register resource
def users_resource_handler() -> str:
    """Return user data."""
    users = [
        {"id": 1, "name": "Alice", "email": "alice@example.com"},
        {"id": 2, "name": "Bob", "email": "bob@example.com"}
    ]
    return json.dumps(users, indent=2)

users_resource_def = ResourceDefinition(
    uri="db://localhost/users",
    name="Users Database",
    description="User records",
    mimeType="application/json"
)

server.register_resource(users_resource_def, users_resource_handler)

print("\\n" + "="*70)
print("MCP SERVER READY")
print("="*70)

## 4. MCP Client

Clients connect to servers and use their tools/resources.

In [ ]:
class MCPClient:
    """
    Simple MCP client implementation.
    
    Connects to servers, discovers tools/resources, executes calls.
    """
    
    def __init__(self, name: str, version: str = "1.0.0"):
        self.name = name
        self.version = version
        self.message_id = 0
        self.connected_servers: Dict[str, Dict] = {}
        
    def get_next_id(self) -> int:
        """Get next message ID."""
        self.message_id += 1
        return self.message_id
    
    def connect(self, server: MCPServer) -> bool:
        """Connect to MCP server."""
        # Send initialize
        init_msg = {
            "jsonrpc": "2.0",
            "id": self.get_next_id(),
            "method": "initialize",
            "params": {
                "protocolVersion": "2024-11-05",
                "capabilities": {
                    "tools": True,
                    "resources": True
                },
                "clientInfo": {
                    "name": self.name,
                    "version": self.version
                }
            }
        }
        
        response = server.handle_message(init_msg)
        
        if "result" in response:
            server_info = response["result"]["serverInfo"]
            self.connected_servers[server_info["name"]] = {
                "server": server,
                "info": server_info,
                "capabilities": response["result"]["capabilities"]
            }
            print(f"✓ Connected to {server_info['name']} v{server_info['version']}")
            return True
        
        return False
    
    def list_tools(self, server_name: str) -> List[Dict]:
        """List tools from connected server."""
        if server_name not in self.connected_servers:
            print(f"Not connected to server: {server_name}")
            return []
        
        server = self.connected_servers[server_name]["server"]
        
        msg = {
            "jsonrpc": "2.0",
            "id": self.get_next_id(),
            "method": "tools/list",
            "params": {}
        }
        
        response = server.handle_message(msg)
        return response.get("result", {}).get("tools", [])
    
    def call_tool(self, server_name: str, tool_name: str, arguments: Dict) -> str:
        """Call tool on server."""
        if server_name not in self.connected_servers:
            return f"Error: Not connected to server '{server_name}'"
        
        server = self.connected_servers[server_name]["server"]
        
        msg = {
            "jsonrpc": "2.0",
            "id": self.get_next_id(),
            "method": "tools/call",
            "params": {
                "name": tool_name,
                "arguments": arguments
            }
        }
        
        response = server.handle_message(msg)
        
        if "result" in response:
            content = response["result"]["content"][0]
            return content["text"]
        elif "error" in response:
            return f"Error: {response['error']['message']}"
        
        return "Unknown error"
    
    def list_resources(self, server_name: str) -> List[Dict]:
        """List resources from server."""
        if server_name not in self.connected_servers:
            print(f"Not connected to server: {server_name}")
            return []
        
        server = self.connected_servers[server_name]["server"]
        
        msg = {
            "jsonrpc": "2.0",
            "id": self.get_next_id(),
            "method": "resources/list",
            "params": {}
        }
        
        response = server.handle_message(msg)
        return response.get("result", {}).get("resources", [])
    
    def read_resource(self, server_name: str, uri: str) -> str:
        """Read resource from server."""
        if server_name not in self.connected_servers:
            return f"Error: Not connected to server '{server_name}'"
        
        server = self.connected_servers[server_name]["server"]
        
        msg = {
            "jsonrpc": "2.0",
            "id": self.get_next_id(),
            "method": "resources/read",
            "params": {"uri": uri}
        }
        
        response = server.handle_message(msg)
        
        if "result" in response:
            content = response["result"]["contents"][0]
            return content["text"]
        elif "error" in response:
            return f"Error: {response['error']['message']}"
        
        return "Unknown error"

# Create client
client = MCPClient(name="my-ai-app", version="1.0.0")

# Connect to server
print("\\nConnecting to server...")
client.connect(server)

# Discover tools
print("\\nAvailable tools:")
tools = client.list_tools("demo-server")
for tool in tools:
    print(f"  • {tool['name']}: {tool['description']}")

# Call tools
print("\\n" + "="*70)
print("TOOL EXECUTION")
print("="*70)

result = client.call_tool("demo-server", "calculate", {"expression": "25 * 4 + 10"})
print(f"\\nCalculate 25 * 4 + 10:")
print(f"  {result}")

result = client.call_tool("demo-server", "get_weather", {"location": "San Francisco"})
print(f"\\nWeather in San Francisco:")
print(f"  {result}")

# Discover resources
print("\\n" + "="*70)
print("RESOURCE ACCESS")
print("="*70)

resources = client.list_resources("demo-server")
print("\\nAvailable resources:")
for resource in resources:
    print(f"  • {resource['name']} ({resource['uri']})")

# Read resource
print("\\nReading users resource:")
users_data = client.read_resource("demo-server", "db://localhost/users")
print(users_data)

## 5. Real-World MCP Servers

**Official MCP Servers** (by Anthropic and community):

1. **Filesystem Server**: Read/write local files
2. **GitHub Server**: Query repos, issues, PRs
3. **PostgreSQL Server**: Query databases
4. **Google Drive Server**: Access Drive files
5. **Slack Server**: Send messages, read channels
6. **Memory Server**: Simple key-value storage

**Custom Servers** you might build:
- CRM integration (Salesforce)
- Internal APIs
- Custom databases
- Proprietary tools

In [ ]:
class FileSystemMCPServer(MCPServer):
    """
    MCP server for filesystem operations.
    
    Exposes file read/write as resources and tools.
    """
    
    def __init__(self, root_dir: str = "./"):
        super().__init__(name="filesystem-server", version="1.0.0")
        self.root_dir = root_dir
        self._register_filesystem_tools()
    
    def _register_filesystem_tools(self):
        """Register filesystem tools."""
        
        # Read file tool
        def read_file_handler(path: str) -> str:
            try:
                full_path = f"{self.root_dir}/{path}"
                with open(full_path, 'r') as f:
                    content = f.read()
                return f"File content ({len(content)} chars):\\n{content[:200]}..."
            except FileNotFoundError:
                return f"File not found: {path}"
            except Exception as e:
                return f"Error reading file: {str(e)}"
        
        read_file_tool = ToolDefinition(
            name="read_file",
            description="Read content of a file",
            inputSchema={
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "File path relative to root"}
                },
                "required": ["path"]
            }
        )
        
        self.register_tool(read_file_tool, read_file_handler)
        
        # List directory tool
        def list_dir_handler(path: str = ".") -> str:
            import os
            try:
                full_path = f"{self.root_dir}/{path}"
                items = os.listdir(full_path)
                return "\\n".join([f"  • {item}" for item in items[:10]])
            except Exception as e:
                return f"Error listing directory: {str(e)}"
        
        list_dir_tool = ToolDefinition(
            name="list_directory",
            description="List files in a directory",
            inputSchema={
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "Directory path"}
                },
                "required": ["path"]
            }
        )
        
        self.register_tool(list_dir_tool, list_dir_handler)

# Example: GitHub MCP Server structure
class GitHubMCPServer(MCPServer):
    """
    MCP server for GitHub integration.
    
    (Simplified example - production would use GitHub API)
    """
    
    def __init__(self, token: Optional[str] = None):
        super().__init__(name="github-server", version="1.0.0")
        self.token = token
        self._register_github_tools()
    
    def _register_github_tools(self):
        """Register GitHub tools."""
        
        # Get repo info
        def get_repo_handler(owner: str, repo: str) -> str:
            # Simulated - would call GitHub API
            return json.dumps({
                "owner": owner,
                "repo": repo,
                "stars": 1250,
                "forks": 245,
                "open_issues": 32
            }, indent=2)
        
        get_repo_tool = ToolDefinition(
            name="get_repo",
            description="Get repository information",
            inputSchema={
                "type": "object",
                "properties": {
                    "owner": {"type": "string", "description": "Repository owner"},
                    "repo": {"type": "string", "description": "Repository name"}
                },
                "required": ["owner", "repo"]
            }
        )
        
        self.register_tool(get_repo_tool, get_repo_handler)
        
        # Search issues
        def search_issues_handler(query: str, repo: str) -> str:
            # Simulated
            return json.dumps([
                {"number": 123, "title": "Bug in feature X"},
                {"number": 124, "title": "Feature request: Y"}
            ], indent=2)
        
        search_issues_tool = ToolDefinition(
            name="search_issues",
            description="Search issues in a repository",
            inputSchema={
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query"},
                    "repo": {"type": "string", "description": "Repository (owner/name)"}
                },
                "required": ["query", "repo"]
            }
        )
        
        self.register_tool(search_issues_tool, search_issues_handler)

print("Example MCP Server Integrations:\\n")

# Filesystem server
print("1. FILESYSTEM SERVER")
print("-" * 50)
fs_server = FileSystemMCPServer(root_dir=".")
print()

# GitHub server
print("2. GITHUB SERVER")
print("-" * 50)
gh_server = GitHubMCPServer(token="dummy_token")
print()

# Connect client to multiple servers
print("="*70)
print("MULTI-SERVER CLIENT")
print("="*70)

multi_client = MCPClient(name="multi-server-client")
multi_client.connect(fs_server)
multi_client.connect(gh_server)

print("\\nAll available tools:")
for server_name in multi_client.connected_servers:
    print(f"\\n{server_name.upper()}:")
    tools = multi_client.list_tools(server_name)
    for tool in tools:
        print(f"  • {tool['name']}")

## 6. MCP with Claude Desktop

**Claude Desktop** has built-in MCP support.

**Setup**:
1. Create MCP server (Python, Node.js, etc.)
2. Configure in `claude_desktop_config.json`
3. Claude automatically discovers tools

**Example config**:
```json
{
  "mcpServers": {
    "filesystem": {
      "command": "npx",
      "args": ["-y", "@modelcontextprotocol/server-filesystem", "/Users/username/Documents"]
    },
    "github": {
      "command": "npx",
      "args": ["-y", "@modelcontextprotocol/server-github"],
      "env": {
        "GITHUB_TOKEN": "your_token_here"
      }
    }
  }
}
```

Claude can then:
- "Read the file analysis.txt"
- "What are open issues in myrepo?"
- "List files in my Documents folder"

In [ ]:
def print_claude_integration_guide():
    """Guide for integrating MCP with Claude."""
    
    print("="*70)
    print("CLAUDE DESKTOP MCP INTEGRATION")
    print("="*70)
    
    print("""
**Step 1: Create MCP Server**

Python server example:
```python
from mcp.server import Server
from mcp.types import Tool

server = Server("my-server")

@server.tool()
def my_tool(arg: str) -> str:
    return f"Result: {arg}"

if __name__ == "__main__":
    server.run()
```

**Step 2: Configure Claude Desktop**

Location: 
  • Mac: ~/Library/Application Support/Claude/claude_desktop_config.json
  • Windows: %APPDATA%\\Claude\\claude_desktop_config.json

Config:
```json
{
  "mcpServers": {
    "my-server": {
      "command": "python",
      "args": ["/path/to/my_server.py"]
    }
  }
}
```

**Step 3: Restart Claude Desktop**

Your tools now appear automatically!

**Step 4: Use in Conversations**

User: "Use my_tool with 'hello'"
Claude: [Calls my_tool('hello')] → "Result: hello"

""")
    
    print("\\n" + "="*70)
    print("POPULAR MCP SERVERS")
    print("="*70)
    
    servers = [
        ("@modelcontextprotocol/server-filesystem", "Read/write files"),
        ("@modelcontextprotocol/server-github", "GitHub integration"),
        ("@modelcontextprotocol/server-postgres", "PostgreSQL queries"),
        ("@modelcontextprotocol/server-google-drive", "Google Drive access"),
        ("@modelcontextprotocol/server-slack", "Slack messaging"),
        ("@modelcontextprotocol/server-memory", "Key-value storage"),
    ]
    
    print("\\nInstall with npx:")
    for package, description in servers:
        print(f"  • {package}")
        print(f"    {description}")
        print()

print_claude_integration_guide()

## 7. Security and Best Practices

**Security Considerations**:

1. **Authentication**: Verify clients
2. **Authorization**: Control tool access
3. **Sandboxing**: Limit tool capabilities
4. **Rate limiting**: Prevent abuse
5. **Input validation**: Sanitize inputs
6. **Audit logging**: Track all operations

In [ ]:
class SecureMCPServer(MCPServer):
    """
    MCP server with security features.
    """
    
    def __init__(self, name: str, api_key: str):
        super().__init__(name, version="1.0.0")
        self.api_key_hash = hashlib.sha256(api_key.encode()).hexdigest()
        self.rate_limits = {}  # client_id -> call count
        self.audit_log = []
        
    def authenticate(self, provided_key: str) -> bool:
        """Verify API key."""
        provided_hash = hashlib.sha256(provided_key.encode()).hexdigest()
        return provided_hash == self.api_key_hash
    
    def check_rate_limit(self, client_id: str, max_calls: int = 100) -> bool:
        """Check if client exceeds rate limit."""
        count = self.rate_limits.get(client_id, 0)
        if count >= max_calls:
            return False
        self.rate_limits[client_id] = count + 1
        return True
    
    def audit_log_entry(self, client_id: str, action: str, details: Dict):
        """Log operation for audit."""
        entry = {
            "timestamp": datetime.now().isoformat(),
            "client_id": client_id,
            "action": action,
            "details": details
        }
        self.audit_log.append(entry)
    
    def validate_input(self, input_value: str, max_length: int = 1000) -> bool:
        """Validate input string."""
        if not isinstance(input_value, str):
            return False
        if len(input_value) > max_length:
            return False
        # Check for malicious patterns
        forbidden = ["rm -rf", "DROP TABLE", "__import__"]
        return not any(pattern in input_value for pattern in forbidden)
    
    def handle_message(self, message: Dict, client_id: str = "unknown", 
                      api_key: str = "") -> Dict:
        """Handle message with security checks."""
        
        # Authenticate
        if not self.authenticate(api_key):
            return {
                "jsonrpc": "2.0",
                "id": message.get("id"),
                "error": {
                    "code": -32001,
                    "message": "Authentication failed"
                }
            }
        
        # Rate limit
        if not self.check_rate_limit(client_id):
            return {
                "jsonrpc": "2.0",
                "id": message.get("id"),
                "error": {
                    "code": -32002,
                    "message": "Rate limit exceeded"
                }
            }
        
        # Audit log
        self.audit_log_entry(client_id, message.get("method", "unknown"), message)
        
        # Validate inputs
        if "params" in message:
            for key, value in message["params"].items():
                if isinstance(value, str) and not self.validate_input(value):
                    return {
                        "jsonrpc": "2.0",
                        "id": message.get("id"),
                        "error": {
                            "code": -32003,
                            "message": f"Invalid input: {key}"
                        }
                    }
        
        # Process message
        return super().handle_message(message)

print("MCP Security Best Practices:\\n")

print("1. AUTHENTICATION")
print("  • Use API keys or OAuth tokens")
print("  • Hash and store keys securely")
print("  • Rotate keys regularly\\n")

print("2. AUTHORIZATION")
print("  • Define tool permissions")
print("  • Implement role-based access")
print("  • Principle of least privilege\\n")

print("3. INPUT VALIDATION")
print("  • Sanitize all inputs")
print("  • Check for injection attacks")
print("  • Enforce length limits\\n")

print("4. RATE LIMITING")
print("  • Per-client call limits")
print("  • Time-window restrictions")
print("  • Graceful degradation\\n")

print("5. AUDIT LOGGING")
print("  • Log all operations")
print("  • Include timestamps and client IDs")
print("  • Monitor for anomalies\\n")

print("6. SANDBOXING")
print("  • Run tools in isolated environments")
print("  • Limit file system access")
print("  • Restrict network calls\\n")

# Example secure server
secure_server = SecureMCPServer(name="secure-server", api_key="my_secret_key")

# Simulate authenticated call
print("\\n" + "="*70)
print("SECURE SERVER DEMO")
print("="*70)

msg = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "initialize",
    "params": {}
}

# Success with correct key
response = secure_server.handle_message(msg, client_id="client-1", api_key="my_secret_key")
print("\\nAuthenticated request:")
print(f"  Success: {'result' in response}")

# Fail with wrong key
response = secure_server.handle_message(msg, client_id="client-2", api_key="wrong_key")
print("\\nUnauthenticated request:")
print(f"  Error: {response.get('error', {}).get('message', 'None')}")

# Show audit log
print("\\nAudit log entries:")
for entry in secure_server.audit_log:
    print(f"  [{entry['timestamp']}] {entry['client_id']}: {entry['action']}")

## Summary

You've mastered Model Context Protocol:
- ✅ MCP architecture (servers expose tools/resources, clients consume)
- ✅ Protocol specification (JSON-RPC messages, tool/resource definitions)
- ✅ Building MCP servers (register tools, handle requests)
- ✅ MCP clients (connect, discover, execute)
- ✅ Real-world integrations (filesystem, GitHub, databases)
- ✅ Claude Desktop integration (automatic tool discovery)
- ✅ Security (authentication, rate limiting, input validation, audit logging)

**Key Insights**:
1. **MCP standardizes** AI-tool integration across applications
2. **Servers** expose capabilities via structured JSON schemas
3. **Clients** (AI apps) automatically discover and use tools
4. **Composability** enables mixing multiple servers
5. **Security** requires authentication, validation, and auditing

**MCP vs Traditional Approaches**:
| Aspect | Traditional | MCP |
|--------|------------|-----|
| Integration | Custom per app | Universal protocol |
| Discovery | Manual configuration | Automatic |
| Types | Untyped strings | JSON Schema |
| Security | Ad-hoc | Built-in patterns |
| Reusability | Low | High |

**Use Cases**:
- **Personal AI**: Access local files, databases
- **Enterprise AI**: Integrate with internal tools (CRM, HR, etc.)
- **Developer tools**: GitHub, Slack, Jira integration
- **Data access**: Read databases, APIs, files
- **Custom workflows**: Build domain-specific servers

**Building Production MCP Servers**:
1. Use official SDK (@modelcontextprotocol/sdk)
2. Define clear JSON schemas for tools
3. Implement robust error handling
4. Add authentication and rate limiting
5. Log all operations for auditing
6. Follow principle of least privilege
7. Document tools comprehensively

**Next Steps**: Build custom MCP servers for your domain and integrate with Claude!

## Exercises

1. **Build CRM MCP Server**: Expose customer data and operations
2. **Multi-server Agent**: Create agent that uses filesystem + GitHub servers
3. **Secure Authentication**: Implement OAuth for MCP server